In [1]:
# ============================================
# 0. Install
# ============================================
!pip -q install --upgrade pip
!pip -q install "unsloth[colab-new]" datasets trl transformers accelerate peft bitsandbytes

# Restart runtime after install if Colab asks for it.

In [2]:
# ============================================
# 1. Imports
# ============================================
import os
import gc
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, set_seed

set_seed(42)

# Make CUDA allocation a bit less fragmentation-prone on Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU total memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU total memory (GB): 14.56


In [3]:
# ============================================
# 2. Strict low-VRAM configuration for 10 GB T4
# ============================================
MODEL_NAME = "unsloth/tinyllama-chat-bnb-4bit"

# Start conservatively. 128 is the safest choice for T4.
MAX_SEQ_LENGTH = 128

# Keep LoRA small at first.
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# Very safe T4 settings
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
LEARNING_RATE = 5e-5
MAX_STEPS = 200
WARMUP_STEPS = 30

OUTPUT_DIR = "tinyllama_indic_sentiment_t4"

# Optional: set this to 96 if you still hit OOM.
# MAX_SEQ_LENGTH = 96

In [4]:
# ============================================
# 3. Small helper: monitor GPU memory
# ============================================
def gpu_mem_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1024**3

def gpu_reserved_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_reserved() / 1024**3

def print_gpu(prefix=""):
    if torch.cuda.is_available():
        print(
            f"{prefix} allocated={gpu_mem_gb():.2f} GB | reserved={gpu_reserved_gb():.2f} GB"
        )

class MemoryCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.cuda.is_available():
            print(
                f"[step {state.global_step}] "
                f"allocated={gpu_mem_gb():.2f} GB | reserved={gpu_reserved_gb():.2f} GB"
            )

In [5]:
# ============================================
# 4. Load model in 4-bit
# ============================================
dtype = None  # Unsloth chooses appropriately; T4 usually ends up on fp16 path

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=dtype,
    load_in_4bit=True,
)

print_gpu("After model load:")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-chat-bnb-4bit as a legacy tokenizer.


After model load: allocated=0.72 GB | reserved=0.73 GB


In [6]:
# ============================================
# 5. Attach LoRA adapters
# ============================================
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # important for T4
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print_gpu("After LoRA wrap:")

Unsloth 2026.4.6 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


After LoRA wrap: allocated=0.74 GB | reserved=0.75 GB


In [7]:
# ============================================
# 6. Load dataset
# ============================================
raw_ds = load_dataset("dhruv0808/indic_sentiment_analyzer", split="train")
print(raw_ds)
print("Columns:", raw_ds.column_names)
print("First example:", raw_ds[0])

Dataset({
    features: ['Sentence', 'Label'],
    num_rows: 131044
})
Columns: ['Sentence', 'Label']
First example: {'Sentence': 'The crisis management team is still assessing the situation and developing a plan.', 'Label': 'Neutral'}


In [8]:
# ============================================
# 7. Auto-detect text and label columns
# ============================================
# This avoids hardcoding and makes the notebook robust to schema changes.

candidate_text_cols = ["text", "sentence", "review", "content", "tweet", "input"]
candidate_label_cols = ["label", "sentiment", "output", "target", "class"]

text_col = None
label_col = None

for c in candidate_text_cols:
    if c in raw_ds.column_names:
        text_col = c
        break

for c in candidate_label_cols:
    if c in raw_ds.column_names:
        label_col = c
        break

# Fallback logic if names are unusual
if text_col is None:
    for c in raw_ds.column_names:
        if raw_ds.features[c].dtype == "string":
            text_col = c
            break

if label_col is None:
    for c in raw_ds.column_names:
        if c != text_col:
            label_col = c
            break

print("Detected text column :", text_col)
print("Detected label column:", label_col)

Detected text column : Sentence
Detected label column: Label


In [9]:
# ============================================
# 8. Safer label normalization
# ============================================
def normalize_label(label):
    """
    Maps numeric or text labels to one of:
    Positive, Negative, Neutral
    Returns None for unusable labels.
    """
    if label is None:
        return None

    numeric_map = {
        0: "Negative",
        1: "Neutral",
        2: "Positive",
        "0": "Negative",
        "1": "Neutral",
        "2": "Positive",
    }

    if label in numeric_map:
        return numeric_map[label]

    s = str(label).strip().lower()

    if s in ["negative", "neg", "label_0"]:
        return "Negative"
    if s in ["neutral", "neu", "label_1"]:
        return "Neutral"
    if s in ["positive", "pos", "label_2"]:
        return "Positive"

    # fallback for a few common alternate schemes
    if s == "-1":
        return "Negative"

    return None

In [10]:
# ============================================
# 9. Filter invalid rows first
# ============================================
def is_valid_example(example):
    text = example[text_col]
    label = example[label_col]

    if text is None:
        return False
    if str(text).strip() == "":
        return False
    if normalize_label(label) is None:
        return False

    return True

filtered_ds = raw_ds.filter(is_valid_example, desc="Filtering invalid rows")

print("Before filtering:", len(raw_ds))
print("After filtering :", len(filtered_ds))
print("Dropped         :", len(raw_ds) - len(filtered_ds))

Before filtering: 131044
After filtering : 131018
Dropped         : 26


In [11]:
# ============================================
# 10. Convert dataset to prompt-completion format
# ============================================
def to_prompt_completion(example):
    text = str(example[text_col]).strip()
    label = normalize_label(example[label_col])

    prompt = (
        "You are a sentiment classifier.\n"
        "Classify the sentiment of the text as Positive, Negative, or Neutral.\n\n"
        f"Text: {text}\n"
        "Answer:"
    )

    # prompt = (
    #     "Classify the sentiment of the following text into exactly one label: "
    #     "Positive, Negative, or Neutral.\n\n"
    #     f"Text: {text}\n"
    #     "Sentiment:"
    # )

    completion = f" {label}"

    return {
        "prompt": prompt,
        "completion": completion,
    }

ds = filtered_ds.map(
    to_prompt_completion,
    remove_columns=filtered_ds.column_names,
    desc="Formatting as prompt-completion",
)

print(ds[0])

Formatting as prompt-completion:   0%|          | 0/131018 [00:00<?, ? examples/s]

{'prompt': 'You are a sentiment classifier.\nClassify the sentiment of the text as Positive, Negative, or Neutral.\n\nText: The crisis management team is still assessing the situation and developing a plan.\nAnswer:', 'completion': ' Neutral'}


In [12]:
from collections import Counter

sample_labels = [raw_ds[i][label_col] for i in range(min(2000, len(raw_ds)))]
print(Counter(map(str, sample_labels)).most_common(20))

[('Positive', 778), ('Negative', 740), ('Neutral', 482)]


In [13]:
# ============================================
# 10. Quick token length sanity check
# ============================================
# This is important: for T4 we want to confirm lengths are small.
# If many examples exceed MAX_SEQ_LENGTH, memory pressure rises.

def estimate_length(example):
    joined = example["prompt"] + example["completion"]
    return {"n_tokens": len(tokenizer(joined, add_special_tokens=True)["input_ids"])}

length_ds = ds.select(range(min(2000, len(ds)))).map(estimate_length)
lengths = length_ds["n_tokens"]

print("Sample token stats:")
print("min   :", min(lengths))
print("mean  :", sum(lengths) / len(lengths))
print("p95   :", sorted(lengths)[int(0.95 * len(lengths))])
print("max   :", max(lengths))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Sample token stats:
min   : 45
mean  : 177.547
p95   : 330
max   : 1036


In [14]:
# ============================================
# 11. Optional filter: drop very long outliers
# ============================================
# This is one of the best ways to enforce a T4 memory budget.
# Keep only examples that fit comfortably.

MAX_ALLOWED_TOKENS = MAX_SEQ_LENGTH

def within_limit(example):
    joined = example["prompt"] + example["completion"]
    n = len(tokenizer(joined, add_special_tokens=True)["input_ids"])
    return n <= MAX_ALLOWED_TOKENS

ds = ds.filter(within_limit, desc="Filtering long samples")
print(ds)
print("Remaining examples:", len(ds))

Filtering long samples:   0%|          | 0/131018 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 41943
})
Remaining examples: 41943


In [15]:
for i in range(10):
    print(ds[i])
    print("-" * 80)

{'prompt': 'You are a sentiment classifier.\nClassify the sentiment of the text as Positive, Negative, or Neutral.\n\nText: The crisis management team is still assessing the situation and developing a plan.\nAnswer:', 'completion': ' Neutral'}
--------------------------------------------------------------------------------
{'prompt': 'You are a sentiment classifier.\nClassify the sentiment of the text as Positive, Negative, or Neutral.\n\nText: இந்த மாதத்திற்கான ஊதியத்தை hr துறை சரியான நேரத்தில் செயலாக்கியது.\nAnswer:', 'completion': ' Neutral'}
--------------------------------------------------------------------------------
{'prompt': 'You are a sentiment classifier.\nClassify the sentiment of the text as Positive, Negative, or Neutral.\n\nText: টুইটারে গ্রাহক পরিষেবা খুব সহায়ক এবং বন্ধুত্বপূর্ণ ছিল।\nAnswer:', 'completion': ' Positive'}
--------------------------------------------------------------------------------
{'prompt': 'You are a sentiment classifier.\nClassify the sentiment

In [16]:
from collections import Counter

labels = [ds[i]["completion"].strip() for i in range(min(len(ds), 5000))]
print(Counter(labels))

Counter({'Negative': 1803, 'Positive': 1693, 'Neutral': 1504})


In [17]:
def tok_len(x):
    return len(tokenizer(x["prompt"] + x["completion"], add_special_tokens=True)["input_ids"])

lengths = [tok_len(ds[i]) for i in range(min(len(ds), 2000))]
print("max:", max(lengths))
print("p95:", sorted(lengths)[int(0.95 * len(lengths))])
print("examples > 96:", sum(l > 96 for l in lengths))

max: 128
p95: 126
examples > 96: 1157


In [18]:
# ============================================
# 12. Trainer config
# ============================================
# Key decisions for 10 GB T4:
# - batch size 1
# - grad accumulation
# - no eval during training
# - packing helps throughput for short examples
# - completion_only_loss keeps supervision tiny
# - paged_adamw_8bit for optimizer memory savings

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,

        # Important for short examples
        packing=True,

        # Prompt-completion training
        completion_only_loss=True,

        # T4-safe training
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,

        # precision
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        # optimizer
        optim="paged_adamw_8bit",

        # reduce overhead
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        eval_strategy="no",
        report_to="none",

        # dataloader settings
        dataloader_num_workers=0,

        # a bit safer on Colab
        seed=42,
    ),
)

trainer.add_callback(MemoryCallback())

print_gpu("Before training:")

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=6):   0%|          | 0/41943 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/41943 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Before training: allocated=0.74 GB | reserved=0.75 GB


In [19]:
# ============================================
# 13. Train
# ============================================
torch.cuda.empty_cache()
gc.collect()

trainer.train()

print_gpu("After training:")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 36,208 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 6,307,840 of 1,106,356,224 (0.57% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.719187
20,0.876521
30,0.367654
40,0.307727
50,0.308362
60,0.304449
70,0.282383
80,0.271737
90,0.281478
100,0.268492


[step 10] allocated=0.79 GB | reserved=0.87 GB
[step 20] allocated=0.79 GB | reserved=0.87 GB
[step 30] allocated=0.79 GB | reserved=0.87 GB
[step 40] allocated=0.79 GB | reserved=0.87 GB
[step 50] allocated=0.79 GB | reserved=0.87 GB
[step 60] allocated=0.79 GB | reserved=0.87 GB
[step 70] allocated=0.79 GB | reserved=0.87 GB
[step 80] allocated=0.79 GB | reserved=0.87 GB
[step 90] allocated=0.79 GB | reserved=0.87 GB
[step 100] allocated=0.79 GB | reserved=0.87 GB
[step 110] allocated=0.79 GB | reserved=0.87 GB
[step 120] allocated=0.79 GB | reserved=0.87 GB
[step 130] allocated=0.79 GB | reserved=0.87 GB
[step 140] allocated=0.79 GB | reserved=0.87 GB
[step 150] allocated=0.79 GB | reserved=0.87 GB
[step 160] allocated=0.79 GB | reserved=0.87 GB
[step 170] allocated=0.79 GB | reserved=0.87 GB
[step 180] allocated=0.79 GB | reserved=0.87 GB
[step 190] allocated=0.79 GB | reserved=0.87 GB
[step 200] allocated=0.79 GB | reserved=0.87 GB
[step 200] allocated=0.79 GB | reserved=0.87 GB
A

In [20]:
def predict_sentiment(text):
    prompt = (
        "Classify the sentiment of the following text into exactly one label: "
        "Positive, Negative, or Neutral.\n\n"
        f"Text: {text}\n"
        "Sentiment:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return decoded.split("Sentiment:")[-1].strip()

In [21]:
# ============================================
# 15. Inference helper
# ============================================
FastLanguageModel.for_inference(model)

def predict_sentiment(text, max_new_tokens=4):
    prompt = (
        "Classify the sentiment of the following text into exactly one label: "
        "Positive, Negative, or Neutral.\n\n"
        f"Text: {text}\n"
        "Sentiment:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

print(predict_sentiment("This product is really good and I liked it a lot."))
print(predict_sentiment("The service was terrible and I want a refund."))
print(predict_sentiment("The movie was okay, nothing special."))

Both `max_new_tokens` (=4) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=4) a

Classify the sentiment of the following text into exactly one label: Positive, Negative, or Neutral.

Text: This product is really good and I liked it a lot.
Sentiment: Positive
Classify the sentiment of the following text into exactly one label: Positive, Negative, or Neutral.

Text: The service was terrible and I want a refund.
Sentiment: Negative


Both `max_new_tokens` (=4) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Classify the sentiment of the following text into exactly one label: Positive, Negative, or Neutral.

Text: The movie was okay, nothing special.
Sentiment: Neutral


In [22]:
# ============================================
# 14. Save adapters and tokenizer
# ============================================
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved to: {OUTPUT_DIR}")

Saved to: tinyllama_indic_sentiment_t4


In [ ]:
# Merge weights and save locally as a full model
# model.save_pretrained_merged("merged_model", tokenizer, save_method = "merged_16bit")

# Push the full merged model to Hugging Face
model.push_to_hub_merged("annavivin/tinyllama-indic-sentiment-full", tokenizer, save_method = "merged_16bit", token = "token_here")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [02:04<00:00, 124.83s/it]


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt-full/model.safetensors:   1%|1         | 31.9MB / 2.20GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:28<00:00, 88.90s/it]


Unsloth: Merge process complete. Saved to `/content/annavivin/tinyllama-indic-sentiment-full`


In [ ]:
# Export to 4-bit quantized GGUF (efficient for 10-15GB GPUs/CPUs)
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method = "q4_k_m")

# Or push directly to Hugging Face Hub in GGUF format
model.push_to_hub_gguf("annavivin/tinyllama-indic-sentiment-gguf-q8", tokenizer, quantization_method = "q8_0", token = "token_here")

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:23<00:00, 83.09s/it]


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../tinyllama-chat.Q8_0.gguf:   0%|          |  546kB / 1.17GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/annavivin/tinyllama-indic-sentiment-gguf-q8
Unsloth: Cleaning up temporary files...


'annavivin/tinyllama-indic-sentiment-gguf-q8'